# JWST NIRCam alignment with JHAT / Gaia

Streamlined notebook. Critical methods live in `jwst123.py` (`jwst_phot`, `query_gaia`,
`calc_dispersion`, `align_jwst_image`, `expand_mask`, `add_bin_dq`).

Based on https://jhat.readthedocs.io/en/latest/examples/plot_b_nircam.html


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy import wcs
from astropy import units as u
from astropy.coordinates import SkyCoord
from jhat import st_wcs_align

from jwst123 import (
    jwst_phot,
    query_gaia,
    calc_dispersion,
    jwst_dispersion,
    run_jhat,
    align_jwst_image,
    expand_mask,
    add_bin_dq,
)
from nbutils import xmatch_common

## Relative alignment

Build a photometric catalog from a reference image, then align a second image to it with JHAT.


In [ ]:
# Update these paths for your reduction directory
workdir = Path('jwstred_temp_gaia')
ref_candidates = list(workdir.glob('*f277w*i2d.fits')) + list(workdir.glob('*f277w*_jhat_i2d.fits'))
ref_image = str(ref_candidates[0]) if ref_candidates else None
ref_image

In [ ]:
refcat, ref_catname = jwst_phot(ref_image)
refcat

In [ ]:
align_candidates = list(workdir.glob('*nrcalong_cal.fits')) + list(workdir.glob('*nrcblong_cal.fits'))
align_image = str(align_candidates[0]) if align_candidates else None
align_image

In [ ]:
wcs_align = st_wcs_align()
wcs_align.run_all(
    align_image,
    telescope='jwst',
    outsubdir=str(workdir),
    refcat_racol='ra',
    refcat_deccol='dec',
    refcat_magcol='mag',
    refcat_magerrcol='dmag',
    overwrite=True,
    d2d_max=1,
    showplots=2,
    refcatname=ref_catname,
    histocut_order='dxdy',
    sharpness_lim=(0.3, 0.9),
    roundness1_lim=(-0.7, 0.7),
    SNR_min=3,
    dmag_max=1.0,
)

## Align to Gaia

Use `align_jwst_image` (preferred) or call JHAT with `refcatname='Gaia'`.


In [ ]:
# High-level wrapper used by the pipeline scripts
outdir = str(workdir)
align_jwst_image(align_image=align_image, outdir=outdir, gaia=True, verbose=True, plot=True)

In [ ]:
# Optional: inspect Gaia–JWST residual dispersion
tb_gaia = query_gaia(ref_image, telescope='jwst', save_file=False)
mean_d, med_d, std_d = calc_dispersion(tb_gaia, ref_catname, plot=True)
mean_d, med_d, std_d

## DQ / breakup-star mask helpers

`expand_mask` and `add_bin_dq` write a `*_masked.fits` product with an expanded BIN_DQ extension.


In [ ]:
# Example:
# cal = str(next(workdir.glob('*nrcblong_cal.fits')))
# masked = add_bin_dq(cal)
# masked